# Neural Network + FGM Pipeline with Epsilon Sweep

This notebook follows the handwritten pipeline and uses the model settings loaded from `MachineLearning\NeuralNetworks\best_params.csv`.

Pipeline steps for each epsilon value:

1. `clean.fit(X_train, y_train)` to train the clean model  
2. `clean.predict(X_test)` and evaluate the clean model on clean test data  
3. Save the clean model  
4. Initialize the FGM attack with the current epsilon  
5. Generate adversarial train and test samples  
6. Keep adversarial labels aligned with the original ground-truth labels  
7. Build combined clean+adversarial train and test sets  
8. Retrain using ART's `AdversarialTrainer` with the FGM attack  
9. Evaluate both the clean model and the adversarially trained model on:
   - clean test data
   - adversarial test data
   - combined clean+adversarial test data

The notebook repeats the full pipeline for `eps in [0.01, 0.03, 0.05, 0.1]` and saves one combined summary table.


In [1]:
# If needed, install once:
# !pip install torch scikit-learn adversarial-robustness-toolbox pandas numpy

import warnings
warnings.filterwarnings("ignore")

import os
import ast
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from art.defences.trainer import AdversarialTrainer


In [2]:
BEST_PARAMS_PATH = Path(r"MachineLearning\NeuralNetworks\best_params.csv")

def parse_hidden_layers(value):
    if isinstance(value, (tuple, list)):
        return tuple(int(v) for v in value)

    text = str(value).strip().strip('"').strip("'")
    if text.startswith("(") or text.startswith("["):
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (tuple, list)):
            return tuple(int(v) for v in parsed)

    return tuple(int(part.strip()) for part in text.split(",") if part.strip())

if not BEST_PARAMS_PATH.exists():
    raise ValueError(f"File not found: {BEST_PARAMS_PATH}")

best_params_df = pd.read_csv(BEST_PARAMS_PATH)
BEST_PARAMS = best_params_df.iloc[0].to_dict()
BEST_PARAMS["hidden_layer_sizes"] = parse_hidden_layers(BEST_PARAMS["hidden_layer_sizes"])
BEST_PARAMS["alpha"] = float(BEST_PARAMS["alpha"])
BEST_PARAMS["learning_rate_init"] = float(BEST_PARAMS["learning_rate_init"])
BEST_PARAMS["batch_size"] = int(BEST_PARAMS["batch_size"])
BEST_PARAMS["max_iter"] = int(BEST_PARAMS["max_iter"])
BEST_PARAMS["early_stopping"] = bool(BEST_PARAMS["early_stopping"])
BEST_PARAMS["n_iter_no_change"] = int(BEST_PARAMS["n_iter_no_change"])
BEST_PARAMS["random_state"] = int(BEST_PARAMS["random_state"])

SEED = BEST_PARAMS["random_state"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEFAULT_DATA_PATH = Path(r"CSVs\newDataset.csv")
RUNS_DIR = Path(r"StandardizedRuns")
RUN_GLOB = "NeuralNet_train_*.csv"

# Optional environment overrides:
# - NN_RUN_PATH
# - MODEL_RUN_PATH
ENV_RUN_PATH = os.environ.get("NN_RUN_PATH") or os.environ.get("MODEL_RUN_PATH")

LABEL_COL = "anomaly"
DROP_COLS = {LABEL_COL, "segment", "train", "sampling"}

TEST_SIZE = 0.75
BATCH_SIZE = BEST_PARAMS["batch_size"]
NB_EPOCHS = BEST_PARAMS["max_iter"]
LR = BEST_PARAMS["learning_rate_init"]
WEIGHT_DECAY = BEST_PARAMS["alpha"]
HIDDEN_LAYER_SIZES = BEST_PARAMS["hidden_layer_sizes"]
ACTIVATION_NAME = str(BEST_PARAMS["activation"]).lower()
SOLVER_NAME = str(BEST_PARAMS["solver"]).lower()
LR_POLICY = str(BEST_PARAMS["learning_rate"]).lower()
EARLY_STOPPING = BEST_PARAMS["early_stopping"]
N_ITER_NO_CHANGE = BEST_PARAMS["n_iter_no_change"]

EPS_VALUES = [0.01, 0.03, 0.05, 0.1]

SAVE_MODELS = False
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_path() -> Path:
    if ENV_RUN_PATH:
        candidate = Path(ENV_RUN_PATH)
        if candidate.exists():
            return candidate
        print(f"[warn] Env path not found: {candidate}")

    if RUNS_DIR.exists():
        candidates = sorted(
            RUNS_DIR.glob(RUN_GLOB),
            key=lambda x: x.stat().st_mtime,
            reverse=True,
        )
        if candidates:
            return candidates[0]

    return DEFAULT_DATA_PATH

DATA_PATH = resolve_data_path()
print(f"Using data source: {DATA_PATH}")
print("Loaded best params:")
print(BEST_PARAMS)

Using data source: StandardizedRuns\NeuralNet_train_clean.csv
Loaded best params:
{'hidden_layer_sizes': (128, 64), 'alpha': 0.0001, 'learning_rate_init': 0.01, 'batch_size': 64, 'activation': 'relu', 'solver': 'adam', 'learning_rate': 'adaptive', 'max_iter': 1000, 'early_stopping': True, 'n_iter_no_change': 20, 'random_state': 100}


In [3]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_cols = [c for c in df.columns if c not in DROP_COLS]
    X = df[feature_cols].to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), feature_cols, scaler

X_train, X_test, y_train, y_test, feature_cols, scaler = load_and_prepare(str(DATA_PATH))

Loaded: StandardizedRuns\NeuralNet_train_clean.csv
Rows=1698, Features=18, Label dist=[1351  347]
Train=(424, 18), Test=(1274, 18)


In [4]:
def get_activation(name: str):
    name = name.lower()
    if name == "relu":
        return nn.ReLU
    if name == "tanh":
        return nn.Tanh
    if name == "logistic":
        return nn.Sigmoid
    raise ValueError(f"Unsupported activation for this notebook: {name}")

class MLP(nn.Module):
    def __init__(self, d_in: int, hidden_layer_sizes=HIDDEN_LAYER_SIZES, activation_name: str = ACTIVATION_NAME):
        super().__init__()

        activation_cls = get_activation(activation_name)
        layers = []
        in_features = d_in

        for hidden_units in hidden_layer_sizes:
            layers.append(nn.Linear(in_features, int(hidden_units)))
            layers.append(activation_cls())
            in_features = int(hidden_units)

        layers.append(nn.Linear(in_features, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_art_classifier(
    d_in: int,
    lr: float = LR,
    weight_decay: float = WEIGHT_DECAY,
    hidden_layer_sizes=HIDDEN_LAYER_SIZES,
    activation_name: str = ACTIVATION_NAME,
):
    model = MLP(
        d_in=d_in,
        hidden_layer_sizes=hidden_layer_sizes,
        activation_name=activation_name,
    )
    criterion = nn.CrossEntropyLoss()


    return PyTorchClassifier(
        model=model,
        loss=criterion,
        optimizer=optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay),
        input_shape=(d_in,),
        nb_classes=2,
        clip_values=(0.0, 1.0),
    )


def predict_labels(art_clf: PyTorchClassifier, X: np.ndarray):
    probs = art_clf.predict(X)
    return np.argmax(probs, axis=1)


def eval_from_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }


def eval_classifier(art_clf: PyTorchClassifier, X: np.ndarray, y: np.ndarray, name: str):
    y_pred = predict_labels(art_clf, X)
    metrics = eval_from_predictions(y, y_pred, name)
    return metrics, y_pred


def save_art_model_state(art_clf: PyTorchClassifier, out_path: Path):
    torch.save(art_clf.model.state_dict(), out_path)
    print(f"Saved model state: {out_path}")


In [5]:

# Run the full handwritten pipeline for each epsilon value
all_results = []

print('Training configuration:')
print({
    'hidden_layer_sizes': HIDDEN_LAYER_SIZES,
    'alpha': WEIGHT_DECAY,
    'learning_rate_init': LR,
    'batch_size': BATCH_SIZE,
    'activation': ACTIVATION_NAME,
    'solver': SOLVER_NAME,
    'learning_rate': LR_POLICY,
    'max_iter': NB_EPOCHS,
    'early_stopping': EARLY_STOPPING,
    'n_iter_no_change': N_ITER_NO_CHANGE,
    'random_state': SEED,
})
print("Epsilon sweep:", EPS_VALUES)


Training configuration:
{'hidden_layer_sizes': (128, 64), 'alpha': 0.0001, 'learning_rate_init': 0.01, 'batch_size': 64, 'activation': 'relu', 'solver': 'adam', 'learning_rate': 'adaptive', 'max_iter': 1000, 'early_stopping': True, 'n_iter_no_change': 20, 'random_state': 100}
Epsilon sweep: [0.01, 0.03, 0.05, 0.1]


In [6]:

for eps in EPS_VALUES:
    print("\n" + "=" * 80)
    print(f"Running handwritten pipeline for eps = {eps}")
    print("=" * 80)

    # Step 1: clean.fit(X_train, y_train) -> trained clean model
    art_clean = make_art_classifier(d_in=X_train.shape[1])
    art_clean.fit(X_train, y_train, batch_size=BATCH_SIZE, nb_epochs=NB_EPOCHS)

    # y_pred = clean.predict(X_test)
    # eval_classifier(clean, X_test, y_pred)
    clean_on_clean, y_pred_clean = eval_classifier(
        art_clean,
        X_test,
        y_test,
        f"clean_model_on_clean_test_eps_{eps}",
    )

    # save clean model
    if SAVE_MODELS:
        eps_tag = str(eps).replace(".", "p")
        save_art_model_state(art_clean, ARTIFACT_DIR / f"nn_clean_model_eps_{eps_tag}.pt")

    # Step 2: initialize FGM and generate adversarial samples
    fgm = FastGradientMethod(estimator=art_clean, eps=eps)

    X_train_adv = fgm.generate(x=X_train)
    X_test_adv = fgm.generate(x=X_test)

    print("Adversarial data generated:")
    print("X_train_adv:", X_train_adv.shape)
    print("X_test_adv:", X_test_adv.shape)

    # If y_train_adv is needed, predict it.
    # The handwritten pipeline notes this as optional.
    y_train_adv_pred = predict_labels(art_clean, X_train_adv)
    y_test_adv_pred = predict_labels(art_clean, X_test_adv)

    # For retraining and evaluation targets, keep the original ground-truth labels.
    y_train_adv = y_train.copy()
    y_test_adv = y_test.copy()

    # Build combined clean + adversarial datasets.
    X_train_combined = np.concatenate([X_train, X_train_adv], axis=0).astype(np.float32)
    y_train_combined = np.concatenate([y_train, y_train_adv], axis=0).astype(np.int64)

    X_test_combined = np.concatenate([X_test, X_test_adv], axis=0).astype(np.float32)
    y_test_combined = np.concatenate([y_test, y_test_adv], axis=0).astype(np.int64)

    print("Combined datasets:")
    print("X_train_combined:", X_train_combined.shape)
    print("X_test_combined:", X_test_combined.shape)

    # Evaluate performance of the clean model on adversarial and combined test data
    clean_on_adv, y_pred_adv_clean_model = eval_classifier(
        art_clean,
        X_test_adv,
        y_test_adv,
        f"clean_model_on_adv_test_eps_{eps}",
    )

    clean_on_combined, y_pred_combined_clean_model = eval_classifier(
        art_clean,
        X_test_combined,
        y_test_combined,
        f"clean_model_on_combined_test_eps_{eps}",
    )

    # Retrain using ART's AdversarialTrainer with the FGM attack
    art_adv = make_art_classifier(d_in=X_train.shape[1])

    adv_trainer = AdversarialTrainer(
        classifier=art_adv,
        attacks=fgm,
        ratio=.55,
    )

    adv_trainer.fit(
        X_train,
        y_train,
        batch_size=BATCH_SIZE,
        nb_epochs=NB_EPOCHS,
    )

    if SAVE_MODELS:
        eps_tag = str(eps).replace(".", "p")
        save_art_model_state(art_adv, ARTIFACT_DIR / f"nn_adversarial_trained_model_eps_{eps_tag}.pt")

    # Test using X_test_adv and the combined clean+adv test set
    adv_trained_on_adv, y_pred_adv = eval_classifier(
        art_adv,
        X_test_adv,
        y_test_adv,
        f"adv_trained_model_on_adv_test_eps_{eps}",
    )

    # Also check whether adversarial training preserved clean performance
    adv_trained_on_clean, y_pred_clean_adv_model = eval_classifier(
        art_adv,
        X_test,
        y_test,
        f"adv_trained_model_on_clean_test_eps_{eps}",
    )

    adv_trained_on_combined, y_pred_combined_adv_model = eval_classifier(
        art_adv,
        X_test_combined,
        y_test_combined,
        f"adv_trained_model_on_combined_test_eps_{eps}",
    )

    for result in [
        clean_on_clean,
        clean_on_adv,
        clean_on_combined,
        adv_trained_on_clean,
        adv_trained_on_adv,
        adv_trained_on_combined,
    ]:
        result["eps"] = eps

    all_results.extend([
        clean_on_clean,
        clean_on_adv,
        clean_on_combined,
        adv_trained_on_clean,
        adv_trained_on_adv,
        adv_trained_on_combined,
    ])



Running handwritten pipeline for eps = 0.01

[clean_model_on_clean_test_eps_0.01] acc=0.9474 f1=0.8607
confusion matrix:
[[1000   14]
 [  53  207]]
              precision    recall  f1-score   support

           0     0.9497    0.9862    0.9676      1014
           1     0.9367    0.7962    0.8607       260

    accuracy                         0.9474      1274
   macro avg     0.9432    0.8912    0.9141      1274
weighted avg     0.9470    0.9474    0.9458      1274

Adversarial data generated:
X_train_adv: (424, 18)
X_test_adv: (1274, 18)
Combined datasets:
X_train_combined: (848, 18)
X_test_combined: (2548, 18)

[clean_model_on_adv_test_eps_0.01] acc=0.8901 f1=0.7417
confusion matrix:
[[933  81]
 [ 59 201]]
              precision    recall  f1-score   support

           0     0.9405    0.9201    0.9302      1014
           1     0.7128    0.7731    0.7417       260

    accuracy                         0.8901      1274
   macro avg     0.8266    0.8466    0.8360      1274
weigh

Adversarial training epochs: 100%|██████████| 1000/1000 [00:19<00:00, 51.25it/s]



[adv_trained_model_on_adv_test_eps_0.01] acc=0.9411 f1=0.8414
confusion matrix:
[[1000   14]
 [  61  199]]
              precision    recall  f1-score   support

           0     0.9425    0.9862    0.9639      1014
           1     0.9343    0.7654    0.8414       260

    accuracy                         0.9411      1274
   macro avg     0.9384    0.8758    0.9026      1274
weighted avg     0.9408    0.9411    0.9389      1274


[adv_trained_model_on_clean_test_eps_0.01] acc=0.9372 f1=0.8305
confusion matrix:
[[998  16]
 [ 64 196]]
              precision    recall  f1-score   support

           0     0.9397    0.9842    0.9615      1014
           1     0.9245    0.7538    0.8305       260

    accuracy                         0.9372      1274
   macro avg     0.9321    0.8690    0.8960      1274
weighted avg     0.9366    0.9372    0.9347      1274


[adv_trained_model_on_combined_test_eps_0.01] acc=0.9392 f1=0.8360
confusion matrix:
[[1998   30]
 [ 125  395]]
              preci

Adversarial training epochs: 100%|██████████| 1000/1000 [00:19<00:00, 51.10it/s]



[adv_trained_model_on_adv_test_eps_0.03] acc=0.9521 f1=0.8710
confusion matrix:
[[1007    7]
 [  54  206]]
              precision    recall  f1-score   support

           0     0.9491    0.9931    0.9706      1014
           1     0.9671    0.7923    0.8710       260

    accuracy                         0.9521      1274
   macro avg     0.9581    0.8927    0.9208      1274
weighted avg     0.9528    0.9521    0.9503      1274


[adv_trained_model_on_clean_test_eps_0.03] acc=0.9451 f1=0.8594
confusion matrix:
[[990  24]
 [ 46 214]]
              precision    recall  f1-score   support

           0     0.9556    0.9763    0.9659      1014
           1     0.8992    0.8231    0.8594       260

    accuracy                         0.9451      1274
   macro avg     0.9274    0.8997    0.9126      1274
weighted avg     0.9441    0.9451    0.9441      1274


[adv_trained_model_on_combined_test_eps_0.03] acc=0.9486 f1=0.8651
confusion matrix:
[[1997   31]
 [ 100  420]]
              preci

Adversarial training epochs: 100%|██████████| 1000/1000 [00:19<00:00, 52.28it/s]



[adv_trained_model_on_adv_test_eps_0.05] acc=0.9466 f1=0.8589
confusion matrix:
[[999  15]
 [ 53 207]]
              precision    recall  f1-score   support

           0     0.9496    0.9852    0.9671      1014
           1     0.9324    0.7962    0.8589       260

    accuracy                         0.9466      1274
   macro avg     0.9410    0.8907    0.9130      1274
weighted avg     0.9461    0.9466    0.9450      1274


[adv_trained_model_on_clean_test_eps_0.05] acc=0.9419 f1=0.8508
confusion matrix:
[[989  25]
 [ 49 211]]
              precision    recall  f1-score   support

           0     0.9528    0.9753    0.9639      1014
           1     0.8941    0.8115    0.8508       260

    accuracy                         0.9419      1274
   macro avg     0.9234    0.8934    0.9074      1274
weighted avg     0.9408    0.9419    0.9408      1274


[adv_trained_model_on_combined_test_eps_0.05] acc=0.9443 f1=0.8548
confusion matrix:
[[1988   40]
 [ 102  418]]
              precision

Adversarial training epochs: 100%|██████████| 1000/1000 [00:19<00:00, 51.76it/s]


[adv_trained_model_on_adv_test_eps_0.1] acc=0.9380 f1=0.8294
confusion matrix:
[[1003   11]
 [  68  192]]
              precision    recall  f1-score   support

           0     0.9365    0.9892    0.9621      1014
           1     0.9458    0.7385    0.8294       260

    accuracy                         0.9380      1274
   macro avg     0.9412    0.8638    0.8957      1274
weighted avg     0.9384    0.9380    0.9350      1274


[adv_trained_model_on_clean_test_eps_0.1] acc=0.9380 f1=0.8378
confusion matrix:
[[991  23]
 [ 56 204]]
              precision    recall  f1-score   support

           0     0.9465    0.9773    0.9617      1014
           1     0.8987    0.7846    0.8378       260

    accuracy                         0.9380      1274
   macro avg     0.9226    0.8810    0.8997      1274
weighted avg     0.9368    0.9380    0.9364      1274


[adv_trained_model_on_combined_test_eps_0.1] acc=0.9380 f1=0.8337
confusion matrix:
[[1994   34]
 [ 124  396]]
              precisio

In [11]:

# Full summary table across all epsilon values
summary_df = pd.DataFrame(all_results)
summary_df = summary_df[["eps", "model_eval", "acc", "f1"]]
summary_df


,eps,model_eval,acc,f1
0,0.01,clean_model_on_clean_test_eps_0.01,0.947410,0.860707
1,0.01,clean_model_on_adv_test_eps_0.01,0.890110,0.741697
2,0.01,clean_model_on_combined_test_eps_0.01,0.918760,0.797654
3,0.01,adv_trained_model_on_clean_test_eps_0.01,0.937206,0.830508
4,0.01,adv_trained_model_on_adv_test_eps_0.01,0.941130,0.841438
5,0.01,adv_trained_model_on_combined_test_eps_0.01,0.939168,0.835979
6,0.03,clean_model_on_clean_test_eps_0.03,0.939560,0.837895
7,0.03,clean_model_on_adv_test_eps_0.03,0.508634,0.323974
8,0.03,clean_model_on_combined_test_eps_0.03,0.724097,0.498216
9,0.03,adv_trained_model_on_clean_test_eps_0.03,0.945055,0.859438


In [10]:

# Save metrics
out_csv = Path(r"Results\NeuralNetworksResults\nn_fgm_handwritten_pipeline_eps_sweep_summary.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")


Saved: Results\NeuralNetworksResults\nn_fgm_handwritten_pipeline_eps_sweep_summary.csv
